# 🔌 03 — Protocols & ports: how a dict and a blockchain fit the same hole

In `02_pydantic_from_zero.ipynb` you met the *shapes* the packages exchange. This
notebook is the other half of `a2a_interfaces`: the **ports** — two `typing.Protocol`
declarations that let the controller run against cardboard today and real
infrastructure tomorrow, with **zero code change**. This is the architecture's central
trick, and by the end you will have performed it yourself.

**What you'll be able to do after this notebook**

- Explain duck typing, nominal typing, and structural typing — and why this repo picks structural
- Read `interfaces/src/a2a_interfaces/ports.py` top to bottom and say why each method is there
- Write a valid chain adapter in ten lines and prove it fits with `isinstance`
- State exactly what `@runtime_checkable` checks — and why rule 7 (shared contract tests) exists because of what it *doesn't*
- Draw this repo's ports-and-adapters hexagon and place every mock and real adapter on it

**You need:** notebooks `00_orientation` – `02_pydantic_from_zero`. No lab, no chain,
no LLM endpoint — this chapter runs fully offline.

**Estimated time:** ~60 minutes.

## 1. The swap problem

The controller's whole job (notebook 08) is answering one question: *"may Ada activate
ticket #7 right now?"* To answer it, the controller must ask something else three
questions: who owns #7, what does the ticket say, and what time is it.

Today, in tests, that "something else" is a Python dict in memory. Tomorrow it is a
real blockchain spoken to over HTTP. The requirement is brutal: **swap one for the
other and change zero lines of controller code.**

Both implementations already exist in this repo. Run the cell: they live in different
packages, were written months apart, and share no ancestor — yet the controller
genuinely cannot tell them apart. The rest of this notebook is about what makes that
possible.

In [ ]:
from e2e.skeleton.fakes import FakeChain     # a dict pretending to be a blockchain (notebook 05)
from chainmcp.reader import ChainReader      # real web3 over HTTP (notebook 07)

for cls in (FakeChain, ChainReader):
    print(f"{cls.__name__:12} lives in {cls.__module__:18} — {cls.__doc__.splitlines()[0]}")

shared = set(FakeChain.__mro__) & set(ChainReader.__mro__)
print("\nancestors they share:", [c.__name__ for c in shared], "← only `object`, like ALL classes")
assert shared == {object}

## 2. Duck typing from zero: "if it quacks like a duck…"

Python's default answer to "can I call `thing.speak()`?" is: *try it and see*. When
that line runs, Python looks up the name `speak` on `thing` — it never asks what class
`thing` is, who its parents are, or whether anyone approved the match. So two
completely unrelated classes work in the same function, as long as each *happens to
have* the method. That is **duck typing**: if it walks like a duck and quacks like a
duck, it's a duck.

Watch one function serve two strangers:

In [ ]:
class Duck:
    def speak(self) -> str:
        return "quack"

class Robot:                       # no relation to Duck whatsoever
    def speak(self) -> str:
        return "BEEP BOOP"

def announce(thing) -> None:       # written against a *hope*: "it'll have .speak()"
    print(f"{type(thing).__name__:6} says → {thing.speak()}")

announce(Duck())
announce(Robot())

**✏️ Your turn 2.1 — add a third stranger to the pond**

`announce` was written against a hope, and two strangers already satisfy it. Write a
third — a `Cat` whose `speak()` returns `"meow"` — and hand it over. Success: three
species served by one function, and the un-commented assert passes.

In [ ]:
class Cat:
    def speak(self) -> str:
        return "TODO"              # TODO: a cat says "meow"

announce(Cat())
# assert Cat().speak() == "meow"   # un-comment once the cat speaks

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
class Cat:
    def speak(self) -> str:
        return "meow"

announce(Cat())
assert Cat().speak() == "meow"
```

`announce` never asks what a `Cat` is — the name lookup at call time is the entire
membership test.
</details>

## 3. The fear: nothing checks the duck before the pond

Duck typing's price: the contract exists only in the author's head. Hand `announce`
something speechless and nothing complains at hand-over — the failure detonates **at
the moment of the call**, possibly hours into a run, deep inside code that is far from
the actual mistake. Look at the exception type, and notice *when* it fires:

In [ ]:
class Teapot:
    pass                           # polite, but speechless

silent = Teapot()                  # handing it over: no complaint here...
try:
    announce(silent)               # ...the boom waits for the .speak() call
except AttributeError as err:
    print("AttributeError:", err)

print("\nThe error names the SYMPTOM (missing attribute), not the crime (wrong object handed over).")
print("We want the check at the border, not in the middle of the job.")

## 4. The classic fix: inheritance (nominal typing)

Most languages answer with **nominal typing** — membership by *name and ancestry*.
In Python that means an `abc.ABC` (abstract base class): an official parent that lists
required methods as `@abstractmethod` (a decorator — you built decorators in 01).
Every implementation must **inherit from it**, and Python refuses to even construct an
incomplete subclass. The failure moves from "at the call" to "at construction" — much
earlier, much closer to the crime:

In [ ]:
from abc import ABC, abstractmethod

class Speaker(ABC):                # the official family
    @abstractmethod
    def speak(self) -> str: ...

class Parrot(Speaker):             # joins the family AND does the work
    def speak(self) -> str:
        return "pieces of eight"

class MimeArtist(Speaker):         # joins the family, forgets the work
    pass

print("Parrot()     →", Parrot().speak())
try:
    MimeArtist()                   # refused at CONSTRUCTION, not at the call
except TypeError as err:
    print("MimeArtist() → TypeError:", err)

**✏️ Your turn 4.1 — train the mime**

`MimeArtist` joined the family but forgot the work — and the abstract debt passes down
to its subclasses. Subclass it as `TrainedMime`, add the missing `speak()`, and watch
construction go from refused to fine. Success: the printed line flips from
`still refused` to the mime speaking.

In [ ]:
class TrainedMime(MimeArtist):     # inherits the family AND the unpaid speak() debt
    pass                           # TODO: pay it — def speak(self) -> str: return "…!"

try:
    print("TrainedMime() →", TrainedMime().speak())
except TypeError as err:
    print("still refused →", err)
# assert isinstance(TrainedMime(), Speaker)   # un-comment once construction succeeds

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
class TrainedMime(MimeArtist):
    def speak(self) -> str:
        return "…!"

print("TrainedMime() →", TrainedMime().speak())
assert isinstance(TrainedMime(), Speaker)
```

The abstract debt survives inheritance until some subclass pays it — only then does
Python agree to construct.
</details>

## 5. Why this repo says no to inheritance

Nominal typing has a hidden demand: every implementation must **announce its
allegiance in its own class statement** — `class FakeChain(EntitlementReaderBase)`.
Two problems for this repo:

- **Import direction.** In 00 you learned that imports flow strictly downward:
  `interfaces` sits at the bottom and imports nothing. If recognition worked by
  ancestry, `interfaces` could only recognize classes that descend from a base it
  knows — so either it imports the adapters' base classes from `netctl`/`chainmcp`
  (an upward import, forbidden), or every adapter everywhere is forced to subclass
  `interfaces`' base. The arrows tangle, or the coupling is total.
- **Ancestry is checked; ability isn't.** With an ABC, a class that already has the
  perfect shape — a ten-liner in this notebook, a class from a library you don't
  control — is rejected until you edit its `class` line. The badge matters more than
  the skill.

The repo wants the opposite deal: the port declares a **shape**, and whoever happens
to have that shape qualifies — *even a class that has never heard of the port*.
First, proof that no adapter in this repo inherits anything:

In [ ]:
from netctl.mock import MockProvisioner

for cls in (FakeChain, ChainReader, MockProvisioner):
    print(f"{cls.__name__:16} inherits from → {[b.__name__ for b in cls.__bases__]}")
    assert cls.__bases__ == (object,)      # no base class = no allegiance declared anywhere

print("\nNo inheritance anywhere — yet the controller can still CHECK them. That's next.")

## 6. `typing.Protocol`: declare the shape, not the family

**Structural typing** is duck typing made checkable: you write the required shape down
*once*, as a `Protocol` class, and any object with matching methods satisfies it — no
inheritance, no registration, no edit to the class. (Capital-P `Protocol` here is a
Python typing feature; it has nothing to do with network protocols like HTTP — that
name clash gets fully untangled in §24.)

The toy below is §2's ducks, upgraded: the same freedom, plus a border check.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class CanSpeak(Protocol):          # the SHAPE: "has a speak() returning str"
    def speak(self) -> str: ...

print("Duck   →", isinstance(Duck(), CanSpeak), "  ← never heard of CanSpeak, still fits")
print("Robot  →", isinstance(Robot(), CanSpeak))
print("Teapot →", isinstance(Teapot(), CanSpeak), " ← caught at the BORDER, before any call")

assert isinstance(Duck(), CanSpeak) and isinstance(Robot(), CanSpeak)
assert not isinstance(Teapot(), CanSpeak)

**✏️ Your turn 6.1 — predict: does the family member fit the shape?**

`Parrot` (§4) went the nominal route — it inherits `Speaker` and has never heard of
`CanSpeak`. Write your prediction as `True`/`False` BEFORE running: does
`isinstance(Parrot(), CanSpeak)` pass? Success: the un-commented assert agrees with
the verdict.

In [ ]:
prediction = None                  # TODO: True or False — commit BEFORE running the check
verdict = isinstance(Parrot(), CanSpeak)
print("your prediction:", prediction, "   actual:", verdict)
# assert prediction == verdict     # un-comment once you've committed

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
prediction = True
verdict = isinstance(Parrot(), CanSpeak)
assert prediction == verdict
```

Structural checks read only the object's surface — ancestry neither helps nor hurts,
and `Parrot` has a `speak`, so it fits.
</details>

## 7. Two details in that cell

- **The `...` body.** `def speak(self) -> str: ...` — those three dots are `Ellipsis`,
  a real Python object standing in for "no body here". A Protocol method is a *stub*:
  the signature is the entire content, and nothing in it ever runs.
- **`@runtime_checkable`** — a class decorator (01 again). A bare `Protocol` serves
  only static type checkers; this decorator is what buys `isinstance()` at runtime.
  Without it, `isinstance` doesn't answer `False` — it refuses to answer at all. And
  since a Protocol is a pure declaration, trying to *instantiate* one is also an
  error. Both breakages, on purpose:

In [ ]:
class Whistler(Protocol):          # note: NO @runtime_checkable
    def whistle(self) -> str: ...

try:
    isinstance(Duck(), Whistler)
except TypeError as err:
    print("isinstance vs plain Protocol → TypeError:", err)

try:
    CanSpeak()                     # a shape is not a thing you can build
except TypeError as err:
    print("CanSpeak()                  → TypeError:", err)

## 8. The real thing: `ports.py`, all of it

Here is the actual contract between the controller's judgment and the outside world —
the *entire file*, under 60 lines, two Protocols: `EntitlementReader` (the chain, read
side only) and `NetworkProvisioner` (the router hands). Read it top to bottom; the
next sections walk it method by method. Notice the module docstring says out loud what
§5 argued: *"any class with the right methods satisfies it, no inheritance required."*

In [ ]:
import inspect
from a2a_interfaces import ports

source = inspect.getsource(ports)
print(source)
assert len(source.splitlines()) < 60   # the whole cross-world contract fits on one screen

**✏️ Your turn 8.1 — harvest the port names mechanically**

You just printed the whole file into `source`. Scan its lines and collect the two
class names — no imports, just string work. Success:
`["EntitlementReader", "NetworkProvisioner"]`, in file order.

In [ ]:
port_names = []
for line in source.splitlines():
    pass                           # TODO: if line.startswith("class "): grab the name before "("

print("found →", port_names)
# assert port_names == ["EntitlementReader", "NetworkProvisioner"]   # un-comment when both appear

<details><summary>✅ Solution 8.1 — peek only after trying</summary>

```python
port_names = []
for line in source.splitlines():
    if line.startswith("class "):
        port_names.append(line.split()[1].split("(")[0])
assert port_names == ["EntitlementReader", "NetworkProvisioner"]
```

The entire cross-world contract is two class statements — small enough to harvest
with a `startswith`.
</details>

## 9. The port surface at a glance

Everything an object must have to *be* one of these, extracted mechanically: four
methods each. This is the complete vocabulary the controller is allowed to use when
talking to a chain or to a router — if a capability isn't named here, the controller
cannot want it.

In [ ]:
from a2a_interfaces.ports import EntitlementReader, NetworkProvisioner

reader_methods = [m for m in dir(EntitlementReader) if not m.startswith("_")]
prov_methods = [m for m in dir(NetworkProvisioner) if not m.startswith("_")]
print("EntitlementReader  →", reader_methods)
print("NetworkProvisioner →", prov_methods)

assert reader_methods == ["chain_time", "get", "owner_of", "watch_revoked"]
assert prov_methods == ["apply_bandwidth", "apply_telemetry", "health", "teardown"]

## 10. `EntitlementReader`: `owner_of` and `get`

- `owner_of(entitlement_id) -> str` — *who holds ticket N right now?* Returns an
  address (the `0x…` account string whose regex guarded it in 02; notebook 06 makes
  addresses real). Ownership can change hands, so reads are always "right now".
- `get(entitlement_id) -> EntitlementView` — the whole ticket, decoded into the frozen
  pydantic model from 02: issuer, validity window, service params, the `revoked` flag.

Signatures, read straight off the port. The quotes around `'int'` and `'str'` are
`from __future__ import annotations` at work (02 met it): annotations are stored as
strings, evaluated only if someone asks.

In [ ]:
print("owner_of", inspect.signature(EntitlementReader.owner_of))
print("get     ", inspect.signature(EntitlementReader.get))

from a2a_interfaces import EntitlementView
print("\nEntitlementView fields:", list(EntitlementView.model_fields))

**✏️ Your turn 10.1 — predict what an annotation is made of**

§10 said the quotes around `'int'` come from postponed annotations. So what does
`inspect.signature(EntitlementReader.get).return_annotation` actually hold — the
`EntitlementView` class, or the string `"EntitlementView"`? Predict `"type"` or
`"str"` before running. Success: the un-commented assert agrees.

In [ ]:
annotation = inspect.signature(EntitlementReader.get).return_annotation
prediction = None                  # TODO: "type" or "str" — what is the annotation's type name?
print("annotation:", repr(annotation), "   its type:", type(annotation).__name__)
# assert prediction == type(annotation).__name__   # un-comment once committed

<details><summary>✅ Solution 10.1 — peek only after trying</summary>

```python
annotation = inspect.signature(EntitlementReader.get).return_annotation
prediction = "str"
assert prediction == type(annotation).__name__
```

`from __future__ import annotations` stores every annotation as unevaluated text —
the quotes in the `repr` are the giveaway.
</details>

## 11. Pause on `chain_time` — a design decision (ADR-004)

Why would a *chain reader* tell the time? Because a ticket's `start_time` and
`end_time` are written in **chain seconds** — `block.timestamp`, stamped by the chain
itself (06 explains blocks). Judging "is the ticket valid *now*?" with your laptop's
clock would compare numbers from two different authorities: OS clocks drift, and in a
fake world time is wherever the test put it. So ADR-004 places the canonical clock
**on the read port**: the authority that stamped the ticket also answers what time it
is. The repo's discipline follows: OS timers may *schedule* wake-ups, but every action
re-checks `chain_time()` before acting.

The decision is written into the port itself — the stub carries a docstring:

In [ ]:
from a2a_interfaces.fixtures import WINDOW

doc = EntitlementReader.chain_time.__doc__
print("chain_time docstring →", doc)
assert "ADR-004" in doc and "canonical clock" in doc   # the decision travels WITH the interface

print("\nticket #7's window:", WINDOW.start, "→", WINDOW.end, " (plain unix seconds — 04 dissects them)")
print("whoever judges validity must compare against these in the CHAIN's seconds, nobody else's")

## 12. `watch_revoked` — a callback port

A **callback** is a function you hand over *as a value*, for someone else to call
later when something happens. (You already treated functions as values in 01 —
decorators are built on exactly that move.) The type hint reads aloud:
`Callable[[int], None]` = "a function taking one `int`, returning nothing".

The port's promise: *when a ticket is revoked, I will call your function with the
ticket id.* That is how a controller learns about a mid-session revocation without
polling in a loop. Toy first — the whole callback idea in ten lines:

In [ ]:
class Doorbell:
    def __init__(self):
        self._listeners = []
    def on_ring(self, callback):           # you REGISTER now...
        self._listeners.append(callback)
    def ring(self):
        for listener in self._listeners:   # ...you get CALLED later
            listener("ding-dong")

def note(sound):
    print("heard:", sound)

bell = Doorbell()
bell.on_ring(note)                         # note the missing (): the function ITSELF is the argument
bell.ring()

print("\nport version: watch_revoked" + str(inspect.signature(EntitlementReader.watch_revoked)))

**✏️ Your turn 12.1 — register a second listener**

A callback port promises: register now, get called later. Lists can listen too —
register `sounds.append` on the same `bell` (no parentheses: the function itself is
the argument), ring, and watch BOTH listeners fire. Success: `heard: ding-dong`
prints again AND `sounds` captures `"ding-dong"`.

In [ ]:
sounds: list[str] = []
# TODO: register it — bell.on_ring(sounds.append)
bell.ring()
print("captured →", sounds)
# assert sounds == ["ding-dong"]   # un-comment once registered

<details><summary>✅ Solution 12.1 — peek only after trying</summary>

```python
sounds: list[str] = []
bell.on_ring(sounds.append)
bell.ring()
assert sounds == ["ding-dong"]
```

A bound method is a value like any other — exactly the move §21 pulls on the real
`watch_revoked`.
</details>

## 13. `NetworkProvisioner`: the two `apply_*` hands

The other port faces the routers. A **session** is the controller's name for one
activation of one ticket; its `session_id` label travels into every call so router
config can be *named after* the session (09 shows why that makes cleanup possible).

- `apply_bandwidth(session_id, path, capacity_bps, qos_class)` — enforce a rate limit
  on a concrete path. `path` is a `ResolvedPath` (02): real device and interface
  names, because rule 6 makes netctl topology-agnostic — it is *told* where, it never
  guesses.
- `apply_telemetry(session_id, target, sensor_paths, collector_endpoint,
  sample_interval_s)` — make a device export measurements toward the buyer's collector.

Both return `ApplyResult(ok, detail)`: failure is *reported as a value*, not raised —
the caller decides what a failed apply means.

In [ ]:
for method in (NetworkProvisioner.apply_bandwidth, NetworkProvisioner.apply_telemetry):
    sig = inspect.signature(method)
    print(f"{method.__name__}{sig}\n")
    assert sig.return_annotation == "ApplyResult"   # stored as a string — postponed annotations again

## 14. `teardown` — the contract travels with the interface

Gloss: an operation is **idempotent** when doing it twice leaves the world exactly as
doing it once — which makes retries free and lets cleanup code run "just in case",
unconditionally. Rule 8 demands this of every teardown in the system. Now look at
*where* the demand is written: in the port's own docstring. Every implementer reads
the law at the exact moment they type the method; every caller may rely on it without
reading any implementation. An interface can carry law, not just shape.

In [ ]:
doc = NetworkProvisioner.teardown.__doc__
print("teardown docstring →", doc)
assert "idempotent" in doc                          # rule 8, stated in the port itself

print("\nhealth", inspect.signature(NetworkProvisioner.health), " ← the 'can you hear me?' probe")

**✏️ Your turn 14.1 — find every law the ports carry**

Two stubs in `ports.py` carry docstring law; the other six are shape only. Sweep both
ports mechanically and collect each `Port.method` whose stub has a docstring.
Success: exactly the ADR-004 clock and the rule-8 teardown.

In [ ]:
lawful = []
for port in (EntitlementReader, NetworkProvisioner):
    for name in [m for m in dir(port) if not m.startswith("_")]:
        pass                       # TODO: if getattr(port, name).__doc__: lawful.append(f"{port.__name__}.{name}")

print("stubs carrying law →", lawful)
# assert sorted(lawful) == ["EntitlementReader.chain_time", "NetworkProvisioner.teardown"]

<details><summary>✅ Solution 14.1 — peek only after trying</summary>

```python
lawful = []
for port in (EntitlementReader, NetworkProvisioner):
    for name in [m for m in dir(port) if not m.startswith("_")]:
        if getattr(port, name).__doc__:
            lawful.append(f"{port.__name__}.{name}")
assert sorted(lawful) == ["EntitlementReader.chain_time", "NetworkProvisioner.teardown"]
```

Only decisions every implementer must honor get written into the port — the other six
stubs are pure shape.
</details>

## 15. Write your own chain adapter: `TinyReader`

Ten lines, answering with the canonical story values (Ada, ticket #7 — every one of
them dissected properly in 04). If structural typing keeps its promise, this
notebook-born class — which never imports the port, never inherits, never registers —
passes the same border check the real adapters pass:

In [ ]:
from a2a_interfaces import fixtures as fx

class TinyReader:
    def owner_of(self, entitlement_id: int) -> str:
        return fx.ADA
    def get(self, entitlement_id: int):
        return fx.CANONICAL_ENTITLEMENT_VIEW
    def chain_time(self) -> int:
        return fx.WINDOW.start + 120           # 14:02 story time, forever
    def watch_revoked(self, callback) -> None:
        pass                                    # nothing here ever gets revoked

tiny = TinyReader()
assert isinstance(tiny, EntitlementReader)
print("isinstance(tiny, EntitlementReader) → True  ✓ you just wrote a valid chain adapter")
print("owner_of(7)  →", tiny.owner_of(7), " ← Ada")
print("chain_time() →", tiny.chain_time(), "        ← inside ticket #7's window")

## 16. Delete a method, lose the badge

The structural check walks every name the Protocol declares and demands each exists on
the object. Remove one — below, `TinyReader` rebuilt *without* `chain_time` — and
`isinstance` flips to `False`. The cell also computes *which* name is missing,
mechanically, by comparing the port's surface with the object's:

In [ ]:
class TinyReaderNoClock:                        # TinyReader minus one method
    def owner_of(self, entitlement_id: int) -> str:
        return fx.ADA
    def get(self, entitlement_id: int):
        return fx.CANONICAL_ENTITLEMENT_VIEW
    def watch_revoked(self, callback) -> None:
        pass

broken = TinyReaderNoClock()
assert not isinstance(broken, EntitlementReader)
required = {m for m in dir(EntitlementReader) if not m.startswith("_")}
print("isinstance(broken, EntitlementReader) → False  ✗")
print("missing →", required - set(dir(broken)))

**✏️ Your turn 16.1 — detonate what the border check just prevented**

`isinstance` said `False` at the border — but nothing physically stops you from using
`broken` anyway. Call the missing method and read which exception fires. Success:
`caught` names the same exception §3's speechless teapot raised.

In [ ]:
caught = None
try:
    pass                           # TODO: call the hole — broken.chain_time()
except AttributeError as err:
    caught = type(err).__name__
print("caught →", caught)
# assert caught == "AttributeError"   # un-comment once it detonates

<details><summary>✅ Solution 16.1 — peek only after trying</summary>

```python
caught = None
try:
    broken.chain_time()
except AttributeError as err:
    caught = type(err).__name__
assert caught == "AttributeError"
```

Skip the border check and you are back in §3 — the failure waits for the call and
names the symptom, not the crime.
</details>

## 17. The honest limit: `isinstance` checks NAMES ONLY

Time for the uncomfortable truth. `@runtime_checkable` verifies that each declared
*name exists* on the object — nothing else. Not the parameters, not the return types,
not whether the answers make any sense. A pathological liar passes the border check
with flying colors:

In [ ]:
class LyingReader:
    def owner_of(self, entitlement_id):
        return 42                               # an address should be "0x…", not a number
    def get(self, entitlement_id):
        return "one soup, extra noodles"        # not an EntitlementView by any stretch
    def chain_time(self):
        return "half past nine"                 # the canonical clock, as interpretive dance
    def watch_revoked(self, callback):
        return "no."

liar = LyingReader()
assert isinstance(liar, EntitlementReader)      # ← it PASSES
print("isinstance(liar, EntitlementReader) → True   (!)")
print("liar.chain_time() →", repr(liar.chain_time()))

## 18. It gets worse — and that's exactly why rule 7 exists

Wrong signatures pass. Missing parameters pass. On Python 3.13, even a plain *string*
sitting where a method should be passes — the runtime check is essentially "does the
attribute exist", full stop:

In [ ]:
class WrongShapes:
    owner_of = "not even a function"            # an attribute, not a method
    def get(self):                              # missing the entitlement_id parameter
        return None
    def chain_time(self, bogus, extra):         # two parameters too many
        return 0
    def watch_revoked(self):
        pass

assert isinstance(WrongShapes(), EntitlementReader)   # still True (verified: Python 3.13)
print("all four names present → isinstance says True, signatures and sense be damned")
print("\nProtocols check SHAPE. Behavior needs TESTS — that is rule 7:")
print("a mock and its real adapter run the SAME contract-test suite, or the mock is a bug.")

**✏️ Your turn 18.1 — predict: does the fraud pass the OTHER border?**

`WrongShapes` fools `EntitlementReader` because all four names exist. Predict
`True`/`False` BEFORE running: does `isinstance(WrongShapes(), NetworkProvisioner)`
pass too? Success: the un-commented assert agrees — and you can say which names
decided it.

In [ ]:
prediction = None                  # TODO: True or False — commit first
actual = isinstance(WrongShapes(), NetworkProvisioner)
print("prediction:", prediction, "   actual:", actual)
# assert prediction == actual     # un-comment once committed

<details><summary>✅ Solution 18.1 — peek only after trying</summary>

```python
prediction = False
actual = isinstance(WrongShapes(), NetworkProvisioner)
assert prediction == actual
```

Each port checks its own four-name vocabulary, and `WrongShapes` has none of
`apply_bandwidth` / `apply_telemetry` / `teardown` / `health` — sloppy as the check
is, it is per-port.
</details>

## 19. Rule 7's receipts: the shared contract suite

Don't take the rule on faith — the file exists.
`netctl/tests/test_provisioner_contract.py` runs every one of its tests against
**both** provisioners: the mock in CI, the real router when the lab is up. Claims that
`isinstance` cannot see — like *"teardown twice succeeds"* — become red tests there.
The cell lists the suite's test names; each one is a behavior promise the shape check
can't make:

In [ ]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())
suite = (ROOT / "netctl" / "tests" / "test_provisioner_contract.py").read_text()

print(suite.splitlines()[0], "\n")               # the suite states rule 7 in its first line
for line in suite.splitlines():
    if line.startswith("def test_"):
        print("  ", line.removeprefix("def ").split("(")[0])

assert "test_teardown_is_idempotent" in suite     # rule 8 checked as BEHAVIOR, not shape

## 20. The payoff parade, chain side

Now the beat this notebook has been building toward. `FakeChain` (a dict with a
hand-crank clock — notebook 05's star) and `ChainReader` (real web3 over HTTP —
notebook 07's star): **both** `EntitlementReader`. One wrinkle: `ChainReader` cannot
be *constructed* without a live chain — but shape lives on the class, so `issubclass`
answers with zero infrastructure. (That trick works because our ports contain only
methods; a data-carrying Protocol would refuse `issubclass`.)

We build the fake for real, sell it ticket #7 (the mechanics are 05's business — here
it just plays the role of "a chain"), and check both:

In [ ]:
from e2e.skeleton.fakes import FakeClock

clock = FakeClock(fx.WINDOW.start + 120)                          # 14:02 story time
chain = FakeChain(clock, balances={fx.ADA: 10**20}, next_id=fx.TICKET_ID)
ticket = chain.fulfill(fx.CANONICAL_SIGNED_OFFER, buyer=fx.ADA)   # Ada buys — details in 05

print("minted ticket:", ticket)
print(f"{'FakeChain instance':19} → EntitlementReader?", isinstance(chain, EntitlementReader), " ✓")
print(f"{'ChainReader class':19} → EntitlementReader?", issubclass(ChainReader, EntitlementReader),
      " ✓ (no chain needed to check)")
assert isinstance(chain, EntitlementReader) and issubclass(ChainReader, EntitlementReader)
print("\nowner_of(7) →", chain.owner_of(7), " == Ada:", chain.owner_of(7) == fx.ADA)

**✏️ Your turn 20.1 — ask the fake for a ticket nobody minted**

The port names the methods but says nothing about how they FAIL. Ask
`chain.owner_of(99)` — ticket #99 was never minted — and read what a dict-backed fake
raises. Success: `caught == "KeyError"`, an error dialect the real chain won't share.

In [ ]:
caught = None
try:
    pass                           # TODO: ask — chain.owner_of(99)
except KeyError:
    caught = "KeyError"
print("caught →", caught, "  ← the dict's error dialect, not the port's")
# assert caught == "KeyError"      # un-comment once it fires

<details><summary>✅ Solution 20.1 — peek only after trying</summary>

```python
caught = None
try:
    chain.owner_of(99)
except KeyError:
    caught = "KeyError"
assert caught == "KeyError"
```

Failure behavior lives beyond the shape check — one more thing only rule 7's shared
contract suite can pin down.
</details>

## 21. The callback fires

§12 promised that `watch_revoked` calls you back — watch it happen on the fake.
A plain list's `append` takes one argument and returns nothing, which makes it exactly
a `Callable[[int], None]`. Register it, revoke ticket #7, and the reader reaches back
into our world with the id:

In [ ]:
revoked_ids: list[int] = []
chain.watch_revoked(revoked_ids.append)     # a bound method is a perfectly good callback

chain.revoke(fx.TICKET_ID)                  # Bell pulls the plug (the showpiece returns in 12)

print("callback received →", revoked_ids)
assert revoked_ids == [fx.TICKET_ID]
assert chain.get(fx.TICKET_ID).revoked      # the ticket itself now says so too
print("get(7).revoked   →", chain.get(fx.TICKET_ID).revoked)

## 22. The payoff parade, network side

Same trick, other port — and this time both implementations are safely constructible
with zero infrastructure: `MockProvisioner()` needs nothing, and `GnmiProvisioner({})`
— the real TLS-gNMI adapter of notebook 09 — happily exists with an empty device map
(it only dials when asked to apply). Bonus sighting: `FakeNet`, the walking skeleton's
original prop, is *literally the same class* as `MockProvisioner` — an alias kept when
the class moved into netctl:

In [ ]:
from e2e.skeleton.fakes import FakeNet
from netctl.provisioner import GnmiProvisioner

mock, real = MockProvisioner(), GnmiProvisioner({})
print(f"{'MockProvisioner()':19} → NetworkProvisioner?", isinstance(mock, NetworkProvisioner), " ✓")
print(f"{'GnmiProvisioner({})':19} → NetworkProvisioner?", isinstance(real, NetworkProvisioner),
      " ✓ (no lab needed to check)")
assert isinstance(mock, NetworkProvisioner) and isinstance(real, NetworkProvisioner)

print("\nFakeNet is MockProvisioner →", FakeNet is MockProvisioner, " ← an alias, not a copy")
assert FakeNet is MockProvisioner

**✏️ Your turn 22.1 — invoke rule 8 with your own hands**

`teardown` is law-bound to be idempotent (§14). Call `mock.teardown("s-yourturn")`
twice and compare the two results. Success: both `ApplyResult`s report `ok=True` —
the second call is a success, not an error.

In [ ]:
first = mock.teardown("s-yourturn")
second = None                      # TODO: same call, second time
print("first  →", first)
print("second →", second)
# assert first.ok and second.ok    # un-comment once both calls run — rule 8, observed

<details><summary>✅ Solution 22.1 — peek only after trying</summary>

```python
first = mock.teardown("s-yourturn")
second = mock.teardown("s-yourturn")
assert first.ok and second.ok
```

The mock pops with a default instead of raising — the law from the port's docstring,
implemented (and pinned by the shared suite's `test_teardown_is_idempotent`).
</details>

## 23. The swap, performed

Section 1's problem, closed. Below is a function written the way the whole controller
is written: its parameter is annotated with the **port**, and its body uses only the
port's four-word vocabulary. We run it against `TinyReader` (born in this notebook)
and `FakeChain` (from a package it has never met) — same function, zero changes.
Swap in `ChainReader` on a live chain and it *still* wouldn't change; notebook 12 does
exactly that.

In [ ]:
def seconds_left(reader: EntitlementReader, ticket_id: int) -> int:
    """How much validity remains — judged by the CHAIN's clock (ADR-004)."""
    view = reader.get(ticket_id)
    return view.end_time - reader.chain_time()

for backend in (tiny, chain):
    print(f"{type(backend).__name__:11} says ticket #7 has {seconds_left(backend, 7)} s left")

assert seconds_left(tiny, 7) == seconds_left(chain, 7) == 7080
print("\nsame function, dict or chain — the swap costs zero lines  ✓")

## 24. This pattern has a name: ports and adapters (hexagonal architecture)

The five-sentence version. (1) Put the decision-making code — the **domain**, here the
controller's predicate (notebook 08) — in the middle, and forbid it from importing
anything that does I/O. (2) For each thing the domain needs from the outside world,
declare a **port**: an interface owned by the domain's side of the wall, saying only
what it needs — our two Protocols. (3) For each real technology, write an **adapter**:
a class that speaks the technology on one side and fits a port on the other —
`ChainReader` adapts web3, `GnmiProvisioner` adapts gNMI. (4) Mocks are simply *more
adapters* into the same ports, which is why tests and production exercise the same
domain code. (5) Draw the domain as a hexagon with ports on its edges and adapters
plugged in from outside — hence the name. This repo's hexagon:

```text
                       the controller's domain (08)
                     ┌────────────────────────────┐
  FakeChain (05)     │                            │     MockProvisioner (09)
  a dict        ────▶│ EntitlementReader (port)   │◀────  two dicts
                     │                            │
                     │      pure judgment,        │
                     │      no I/O imports        │
                     │                            │
  ChainReader (07)   │ NetworkProvisioner (port)  │     GnmiProvisioner (09)
  web3 over HTTP ───▶│                            │◀────  TLS gNMI to routers
                     └────────────────────────────┘
              both ports live in a2a_interfaces.ports;
          adapters satisfy them STRUCTURALLY (this notebook)
```

Two name clashes, disarmed for the rest of the course:

- **port (architecture) vs port (network).** Here, a *port* is a hole in the domain's
  wall — an interface the domain owns. In networking, a *port* is a number a service
  listens on (the lab routers speak gNMI on port 57400). Same word, unrelated ideas.
- **`Protocol` (typing) vs protocol (network).** Capital-P `typing.Protocol` is
  Python's structural-interface feature — this notebook. Lowercase network *protocol*
  is an agreed message format two machines speak (HTTP, gNMI — notebook 09). Also
  unrelated.

One last cell: the whole hexagon, machine-checked — five adapters (including yours)
against two ports, with no chain and no lab anywhere in sight:

In [ ]:
pairs = [
    ("FakeChain",          FakeChain,        EntitlementReader),
    ("ChainReader",        ChainReader,      EntitlementReader),
    ("TinyReader (yours)", TinyReader,       EntitlementReader),
    ("MockProvisioner",    MockProvisioner,  NetworkProvisioner),
    ("GnmiProvisioner",    GnmiProvisioner,  NetworkProvisioner),
]
print("adapter                 fits port")
for name, cls, port in pairs:
    fits = issubclass(cls, port)
    print(f"  {name:20} → {port.__name__:19} {'✓' if fits else '✗'}")
    assert fits

print("\nfive adapters, two ports, zero inheritance — the hexagon holds")

**✏️ Your turn 24.1 — predict: are the hexagon's two holes interchangeable?**

Every adapter fits its port. Predict `True`/`False` BEFORE running: does `FakeChain`
— a flawless `EntitlementReader` — also fit `NetworkProvisioner`? Success: the
un-commented assert agrees, and you can explain the verdict using the two four-word
vocabularies from §9.

In [ ]:
prediction = None                  # TODO: True or False — commit first
actual = issubclass(FakeChain, NetworkProvisioner)
print("prediction:", prediction, "   actual:", actual)
# assert prediction == actual      # un-comment once committed

<details><summary>✅ Solution 24.1 — peek only after trying</summary>

```python
prediction = False
actual = issubclass(FakeChain, NetworkProvisioner)
assert prediction == actual
```

The two ports share no method names, so each edge of the hexagon is a
differently-shaped hole — fitting one says nothing about the other.
</details>

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| Called `.speak()` on two unrelated classes | duck typing — names are looked up at call time | every Python line you'll ever read |
| Watched `AttributeError` detonate mid-call | duck typing has no border check | the motivation for ports |
| Got an incomplete ABC subclass refused | nominal typing — membership by ancestry | the road this repo did *not* take (§5) |
| `isinstance(TinyReader(), EntitlementReader)` | structural typing via `typing.Protocol` + `@runtime_checkable` | how every adapter joins this system |
| Read `ports.py` whole; paused on two docstrings | `chain_time` as the canonical clock (ADR-004); idempotency law inside the port (rule 8) | 08 builds the controller on exactly these |
| Fooled `isinstance` with `LyingReader` | shape ≠ behavior — names only | rule 7's shared contract suites (09) |
| Checked 5 adapters against 2 ports, offline | ports and adapters / hexagonal architecture | 11 composes whole worlds by swapping adapters |

## Experiments to try

Predict first, then run.

- Add a fifth method to `TinyReader` — say `def brew_coffee(self): return "☕"`.
  Predict `isinstance` before running: do EXTRA methods matter to a Protocol?
- Write a `TinyProvisioner` against `NetworkProvisioner` from scratch (four methods
  returning `ApplyResult(ok=True)`) and make `isinstance` pass. Then make its
  `teardown` raise `KeyError` on the second call. Does `isinstance` notice? What
  *would* notice? (§19 has the answer's filename.)
- Break it: in §12's `Doorbell`, register a callback that itself raises, then
  `ring()`. Who crashes — the ringer or the listener? What does that say about how
  defensively a `watch_revoked` implementation must treat *your* callback?
- The subtle one: run `isinstance(tiny, EntitlementReader)` (True), then
  `del TinyReader.chain_time`, then check again. Python caches protocol verdicts per
  class — did the answer change when the truth did?

## Check yourself

1. A colleague says "make `FakeChain` inherit from `EntitlementReader`, so it's
   official." What do you tell them — and which CLAUDE.md-style concern (import
   direction, coupling) backs you up?
2. Why does the canonical clock live on the *read port* instead of the controller
   calling `time.time()`? (Two words: whose seconds?)
3. `isinstance(obj, SomePort)` returns `True`. Name two important things you still do
   **not** know about `obj` — and the repo mechanism that closes that gap.
4. What exactly breaks if you remove `@runtime_checkable` from `EntitlementReader` —
   every adapter, or something narrower?
5. "The router listens on port 57400" and "`NetworkProvisioner` is a port" — explain
   each use of the word.

**Where this goes next:** `04_the_canonical_example.ipynb` 🎭 — the fixture values
`TinyReader` parroted (Ada, ticket #7, 10 TOK, the ABI params blob) get dissected byte
by byte.

**Deeper dive:** `e2e/notebooks/netctl_explore.ipynb` and
`e2e/notebooks/skeleton_explore.ipynb` (adapters at work on both sides of the ports) ·
[`docs/03-interfaces.md`](../../../docs/03-interfaces.md) §4–§5 (the ports' spec) ·
[`docs/02-architecture.md`](../../../docs/02-architecture.md) (the import-direction map) ·
[`docs/adr/004-chain-time-canonical-clock.md`](../../../docs/adr/004-chain-time-canonical-clock.md)
(why chain time is the only clock).